# SoftwareX revision: option sensitivity on new demonstration cities

Answers Reviewer 2 (comments 1, 3, 4) with app runs on eight mid-size US cities.
Pipeline functions are an independent copy of the app's Earth Engine processing (logic unchanged in v1.2).

| Run | Setting changed (all else = app defaults: Census 2023 place, Recommended buffer, NLCD, 2000–2024) |
|---|---|
| `B0_default` | none: JJA hottest month, Terra + Aqua |
| `T1_terra` | Terra only |
| `Q1_aqua` | Aqua only (2003–2024; Aqua has no full summer before 2003) |
| `F6_june`, `F7_july`, `F8_aug` | fixed month, Terra + Aqua; their per-year average = **JJA seasonal mean** |
| `V1_no_class21` | vegetated reference without NLCD 21 (Developed, Open Space): 41, 42, 43, 52, 71 |

Trend statistics are computed on the **unsmoothed** annual series: Theil–Sen slope, original Mann–Kendall p, and
Hamed–Rao modified Mann–Kendall p (autocorrelation-corrected). The app's former smoothed-series p is shown for contrast.
Re-running skips finished runs.

In [ ]:
# pip install earthengine-api geemap geopandas pymannkendall requests
import io, os, zipfile, tempfile
from datetime import datetime
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import geemap
import ee
from scipy import stats
try:
    import pymannkendall as mk
    _HAS_MK = True
except Exception:
    _HAS_MK = False

EE_PROJECT = "YOUR-GEE-PROJECT-ID"   # <- your Earth Engine Cloud project
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready; pymannkendall:", _HAS_MK)

In [ ]:
import random
START_YEAR, END_YEAR = 2000, 2024
JJA = ['06-Jun', '07-Jul', '08-Aug']
CA, TO, AO = "Terra + Aqua (combined)", "Terra only (MOD)", "Aqua only (MYD)"
RUNS = {
    "B0_default": dict(platform=CA, months=JJA,         mcd=False, start=2000),
    "T1_terra":   dict(platform=TO, months=JJA,         mcd=False, start=2000),
    "Q1_aqua":    dict(platform=AO, months=JJA,         mcd=False, start=2003),
    "F6_june":    dict(platform=CA, months=['06-Jun'],  mcd=False, start=2000),
    "F7_july":    dict(platform=CA, months=['07-Jul'],  mcd=False, start=2000),
    "F8_aug":     dict(platform=CA, months=['08-Aug'],  mcd=False, start=2000),
    "V1_no_class21": dict(platform=CA, months=JJA,      mcd=False, start=2000, veg=[41, 42, 43, 52, 71]),
}
for r in RUNS.values(): r.setdefault('veg', None); r.update(buffer="Recommended")

# Random draw (fixed seed, documented) from mid-size US cities
POOL = [("Tallahassee","Florida"), ("Chattanooga","Tennessee"), ("Rockford","Illinois"), ("Ogden","Utah"),
        ("Gainesville","Florida"), ("Topeka","Kansas"), ("Bloomington","Indiana"), ("Reading","Pennsylvania")]
SET_A = random.Random(2026).sample(POOL, 4)          # Set A: random draw (already run)

# Set B: purposive, one city per Koppen climate type NOT covered by Set A (Set A = humid subtropical Cfa
# and humid continental Dfa). Rule fixed before running: mid-size US city.
SET_B = [("St. George", "Utah"),        # BWh/BSk hot arid
         ("Modesto", "California"),     # Csa Mediterranean, hot summer
         ("Bellingham", "Washington"),  # Csb/Cfb marine, mild summer
         ("Grand Junction", "Colorado")]# BSk cold semi-arid
SET = "B"                                            # <- "A" or "B"
CITIES = SET_A if SET == "A" else SET_B
print("Set", SET, "cities:", CITIES)

OUT_DIR = "runs"; SUM_DIR = "."
os.makedirs(OUT_DIR, exist_ok=True)
STATE_FIPS = {'California': '06', 'Colorado': '08', 'Washington': '53', 'Florida': '12', 'Tennessee': '47', 'Illinois': '17', 'Utah': '49', 'Kansas': '20', 'Indiana': '18', 'Pennsylvania': '42'}


## App functions (copied from `App_patched.py`)

In [ ]:
# ---------- AOI: load_city_aoi_from_census (app line 696) ----------
def load_city_aoi_from_census(state_fips: str, city_name: str):
    url = f"https://www2.census.gov/geo/tiger/TIGER2023/PLACE/tl_2023_{state_fips.zfill(2)}_place.zip"
    r = requests.get(url, timeout=30); r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        with tempfile.TemporaryDirectory() as tmp:
            z.extractall(tmp)
            shp = [f for f in os.listdir(tmp) if f.endswith(".shp")][0]
            gdf = gpd.read_file(os.path.join(tmp, shp))
    gdf_sel = gdf[gdf["NAME"] == city_name]
    if gdf_sel.empty:
        raise RuntimeError(f"City '{city_name}' not found in Census 2023 Places for this state.")
    gdf_ll = gdf_sel.to_crs(epsg=4326)
    poly = gdf_ll.geometry.iloc[0]
    lon, lat = poly.centroid.x, poly.centroid.y
    zone = int(((lon + 180) / 6) % 60) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    gdf_proj = gdf_ll.to_crs(epsg=epsg)
    aoi_ee = geemap.geopandas_to_ee(gdf_proj)
    return aoi_ee, gdf_ll, epsg

# ---------- buffer helpers (app lines 1076-1113) ----------
def load_nlcd_urban():
    nlcd2019 = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").filter(
        ee.Filter.eq('system:index', '2019')).first().select('landcover')
    return nlcd2019.gte(22).And(nlcd2019.lte(24))

def compute_urban_area_km2(aoi_geom):
    try:
        urban = load_nlcd_urban()
        area_img = ee.Image.pixelArea().rename('area').updateMask(urban)
        stats_ = area_img.reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi_geom,
                                       scale=30, maxPixels=1e9, bestEffort=True)
        area_m2 = ee.Number(stats_.get('area')).getInfo()
        return 0.0 if area_m2 is None else float(area_m2) / 1e6
    except Exception:
        return 0.0

def classify_city_and_buffer(urban_area_km2):
    if urban_area_km2 < 150:   return 5.0, "small city"
    elif urban_area_km2 < 500: return 10.0, "mid-size city"
    else:                      return 15.0, "large city"

# ---------- City AOI exactly as the app builds it (app lines 1966-2033 and 2118-2126) ----------
def build_city_aoi(city, state_fips, buffer_mode="Recommended"):
    city_fc_ee, _, _ = load_city_aoi_from_census(state_fips, city)
    base_geom = city_fc_ee.geometry()                         # "Census boundary"
    urban_area_km2 = compute_urban_area_km2(base_geom)
    rec_km, _cat = classify_city_and_buffer(urban_area_km2)
    buf_km = rec_km if buffer_mode == "Recommended" else 0.0   # app: "No buffer" -> 0
    final_geom = base_geom.buffer(buf_km * 1000) if buf_km > 0 else base_geom
    try:
        gj = final_geom.getInfo()
    except Exception:
        gj = base_geom.getInfo()
    try:
        ee_geom = geemap.geojson_to_ee(gj).geometry()
    except Exception:
        ee_geom = ee.Geometry(gj)
    return ee.FeatureCollection([ee.Feature(ee_geom)]), urban_area_km2, buf_km

In [ ]:
# ---------- NLCD (app lines 878-926, 947-969) ----------
def get_best_nlcd_for_year(year):
    yrs = [2001, 2004, 2006, 2008, 2011, 2013, 2016, 2019, 2021]
    target = min(yrs, key=lambda y: abs(y - year))
    if target <= 2019:
        nlcd = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").filter(
            ee.Filter.eq('system:index', str(target))).first().select('landcover')
    else:
        try:
            nlcd = ee.ImageCollection("USGS/NLCD_RELEASES/2021_REL/NLCD").filter(
                ee.Filter.eq('system:index', '2021')).first().select('landcover')
        except Exception:
            nlcd = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").filter(
                ee.Filter.eq('system:index', '2019')).first().select('landcover')
            target = 2019
    return nlcd, target

def nlcd_masks(nlcd_img, custom_veg_codes=None, custom_urban_codes=None):
    if custom_urban_codes:
        urban = nlcd_img.eq(custom_urban_codes[0])
        for code in custom_urban_codes[1:]:
            urban = urban.Or(nlcd_img.eq(code))
    else:
        urban = nlcd_img.gte(22).And(nlcd_img.lte(24))
    if custom_veg_codes:
        veg = nlcd_img.eq(custom_veg_codes[0])
        for code in custom_veg_codes[1:]:
            veg = veg.Or(nlcd_img.eq(code))
    else:
        forest = nlcd_img.gte(41).And(nlcd_img.lte(43))
        shrub  = nlcd_img.eq(52)
        grass  = nlcd_img.eq(71)
        open_s = nlcd_img.eq(21)
        veg = forest.Or(shrub).Or(grass).Or(open_s)
    return urban, veg

def _lc_percentages(aoi_geom, urban_mask, veg_mask, scale):
    area_img = ee.Image.pixelArea().rename('area')
    total_area = area_img.clip(aoi_geom).reduceRegion(ee.Reducer.sum(), aoi_geom, scale, maxPixels=1e9, bestEffort=True).get('area')
    urban_area = area_img.updateMask(urban_mask).reduceRegion(ee.Reducer.sum(), aoi_geom, scale, maxPixels=1e9, bestEffort=True).get('area')
    veg_area   = area_img.updateMask(veg_mask).reduceRegion(ee.Reducer.sum(), aoi_geom, scale, maxPixels=1e9, bestEffort=True).get('area')
    try:
        tot = float(ee.Number(total_area).getInfo() or 0.0)
        u   = float(ee.Number(urban_area).getInfo() or 0.0)
        v   = float(ee.Number(veg_area).getInfo() or 0.0)
        if tot <= 0: return 0.0, 0.0, 0.0
        up, vp = u / tot * 100.0, v / tot * 100.0
        return up, vp, max(0.0, 100.0 - up - vp)
    except Exception:
        return 0.0, 0.0, 0.0

def landcover_percentages(aoi_geom, nlcd_img, custom_veg_codes=None, custom_urban_codes=None):
    u, v = nlcd_masks(nlcd_img, custom_veg_codes, custom_urban_codes); return _lc_percentages(aoi_geom, u, v, 30)

# ---------- MCD12Q1 (app lines 975-1050) ----------
_MCD_LAST = None
def _mcd12q1_last_year():
    global _MCD_LAST
    if _MCD_LAST is None:
        try:
            t = ee.ImageCollection("MODIS/061/MCD12Q1").aggregate_max('system:time_start').getInfo()
            _MCD_LAST = datetime.utcfromtimestamp(t / 1000.0).year
        except Exception:
            _MCD_LAST = 2023
    return _MCD_LAST

def get_mcd12q1_for_year(year):
    actual_year = max(2001, min(year, _mcd12q1_last_year()))
    img = (ee.ImageCollection("MODIS/061/MCD12Q1")
           .filter(ee.Filter.calendarRange(actual_year, actual_year, 'year'))
           .first().select('LC_Type1'))
    return img, actual_year

def mcd12q1_masks(lc_img, custom_veg_codes=None, custom_urban_codes=None):
    urban = lc_img.eq(13)                        # Urban and Built-up
    veg = lc_img.gte(1).And(lc_img.lte(10))      # IGBP 1-10
    return urban, veg

def landcover_percentages_mcd(aoi_geom, lc_img, *_):
    u, v = mcd12q1_masks(lc_img); return _lc_percentages(aoi_geom, u, v, 500)

In [ ]:
# ---------- MODIS LST pipeline (app lines 1403-1577) ----------
def _scale_mask_modis_lst(img, lst_band, qc_band):
    qc = img.select(qc_band)
    good = qc.bitwiseAnd(3).lt(2)
    return img.select(lst_band).updateMask(good).multiply(0.02).subtract(273.15).rename('LST')

def _modis_platform_flags(platform):
    p = (platform or "").lower()
    if p.startswith("terra only"): return True, False
    if p.startswith("aqua only"):  return False, True
    return True, True

def _modis_label(platform):
    t, a = _modis_platform_flags(platform)
    if t and a: return 'MODIS Terra+Aqua'
    return 'MODIS Terra' if t else 'MODIS Aqua'

def _modis_lst_median(year, month, aoi, lst_band, qc_band, platform):
    start = ee.Date.fromYMD(year, month, 1); end = start.advance(1, 'month')
    use_terra, use_aqua = _modis_platform_flags(platform)
    cols = []
    if use_terra:
        cols.append(ee.ImageCollection('MODIS/061/MOD11A1').filterDate(start, end)
                    .map(lambda i: _scale_mask_modis_lst(i, lst_band, qc_band)))
    if use_aqua:
        cols.append(ee.ImageCollection('MODIS/061/MYD11A1').filterDate(start, end)
                    .map(lambda i: _scale_mask_modis_lst(i, lst_band, qc_band)))
    merged = cols[0] if len(cols) == 1 else cols[0].merge(cols[1])
    return merged.median().clip(aoi)

def modis_lst_day_median(year, month, aoi, platform):
    return _modis_lst_median(year, month, aoi, 'LST_Day_1km', 'QC_Day', platform)

def modis_lst_night_median(year, month, aoi, platform):
    return _modis_lst_median(year, month, aoi, 'LST_Night_1km', 'QC_Night', platform)

def modis_indices_median(year, month, aoi, platform):
    start = ee.Date.fromYMD(year, month, 1); end = start.advance(1, 'month')
    use_terra, _ = _modis_platform_flags(platform)
    _sr_id = 'MODIS/061/MOD09GA' if use_terra else 'MODIS/061/MYD09GA'
    sr = (ee.ImageCollection(_sr_id).filterDate(start, end).filterBounds(aoi)
          .select(['sur_refl_b01', 'sur_refl_b02', 'sur_refl_b06'], ['red', 'nir', 'swir1'])
          .map(lambda img: img.multiply(0.0001)))
    def add_index(img):
        ndvi = img.normalizedDifference(['nir', 'red']).rename('NDVI')
        ndmi = img.normalizedDifference(['nir', 'swir1']).rename('NDMI')
        ndbi = img.normalizedDifference(['swir1', 'nir']).rename('NDBI')
        return ee.Image.cat([ndvi, ndmi, ndbi])
    return sr.map(add_index).median().clip(aoi)

def _lc_for_year(year, use_mcd12q1, veg_codes=None):
    if use_mcd12q1:
        lc_img, lc_year = get_mcd12q1_for_year(year)
        u, v = mcd12q1_masks(lc_img); return lc_img, lc_year, u, v, f"MCD12Q1-{lc_year}"
    lc_img, lc_year = get_best_nlcd_for_year(year)
    u, v = nlcd_masks(lc_img, veg_codes); return lc_img, lc_year, u, v, f"NLCD-{lc_year}"

def process_modis_day_and_best_month(year, aoi, months, use_mcd12q1=False, platform="Terra + Aqua (combined)", veg_codes=None):
    month_numbers = [int(m.split('-')[0]) for m in months]
    temps, lst_by_m = [], {}
    for m in month_numbers:
        lst_img = modis_lst_day_median(year, m, aoi, platform)
        try:
            t = lst_img.reduceRegion(ee.Reducer.median(), aoi, 1000, maxPixels=1e9, bestEffort=True).get('LST').getInfo()
        except Exception:
            t = None
        if t is not None:
            temps.append((t, m)); lst_by_m[m] = lst_img
    if not temps:
        return None, None
    _, best_m = max(temps, key=lambda x: x[0])
    index_img = modis_indices_median(year, best_m, aoi, platform)
    s = index_img.reduceRegion(ee.Reducer.median(), aoi, 1000, maxPixels=1e9, bestEffort=True).getInfo()
    ndvi, ndmi, ndbi = s.get('NDVI'), s.get('NDMI'), s.get('NDBI')

    lc_img, lc_year, urban_m, veg_m, lc_label = _lc_for_year(year, use_mcd12q1, veg_codes)
    lst_img = lst_by_m[best_m]
    urban_t    = lst_img.updateMask(urban_m).reduceRegion(ee.Reducer.median(), aoi, 1000, maxPixels=1e9, bestEffort=True).get('LST').getInfo()
    rural_t    = lst_img.updateMask(veg_m).reduceRegion(ee.Reducer.median(), aoi, 1000, maxPixels=1e9, bestEffort=True).get('LST').getInfo()
    aoi_mean_t = lst_img.reduceRegion(ee.Reducer.mean(), aoi, 1000, maxPixels=1e9, bestEffort=True).get('LST').getInfo()
    if (rural_t is None) or (urban_t is None) or (aoi_mean_t is None):
        return None, None
    pct = landcover_percentages_mcd if use_mcd12q1 else landcover_percentages
    up, vp, op = pct(aoi.geometry(), lc_img, veg_codes) if not use_mcd12q1 else pct(aoi.geometry(), lc_img)
    rec = {'Year': year, 'Month': f"{best_m:02d}", 'Satellite': _modis_label(platform), 'LC_Source': lc_label,
           'Image_Date': f"{year}-{best_m:02d}-15",
           'Urban': round(urban_t, 2), 'Vegetative': round(rural_t, 2), 'UHI': round(urban_t - rural_t, 2),
           'Mean_LST': round(aoi_mean_t, 2),
           'Mean_NDVI': round(ndvi, 4) if ndvi is not None else None,
           'Mean_NDMI': round(ndmi, 4) if ndmi is not None else None,
           'Mean_NDBI': round(ndbi, 4) if ndbi is not None else None,
           'Urban_Percent': up, 'Vegetative_Percent': vp, 'Other_Percent': op, 'LC_Year': lc_year}
    return rec, best_m

def process_modis_night_for_month(year, aoi, best_m, use_mcd12q1=False, platform="Terra + Aqua (combined)", veg_codes=None):
    lc_img, lc_year, urban_m, veg_m, lc_label = _lc_for_year(year, use_mcd12q1, veg_codes)
    lst_img_n  = modis_lst_night_median(year, best_m, aoi, platform)
    urban_t    = lst_img_n.updateMask(urban_m).reduceRegion(ee.Reducer.median(), aoi, 1000, maxPixels=1e9, bestEffort=True).get('LST').getInfo()
    rural_t    = lst_img_n.updateMask(veg_m).reduceRegion(ee.Reducer.median(), aoi, 1000, maxPixels=1e9, bestEffort=True).get('LST').getInfo()
    aoi_mean_t = lst_img_n.reduceRegion(ee.Reducer.mean(), aoi, 1000, maxPixels=1e9, bestEffort=True).get('LST').getInfo()
    if (rural_t is None) or (urban_t is None) or (aoi_mean_t is None):
        return None
    pct = landcover_percentages_mcd if use_mcd12q1 else landcover_percentages
    up, vp, op = pct(aoi.geometry(), lc_img, veg_codes) if not use_mcd12q1 else pct(aoi.geometry(), lc_img)
    return {'Year': year, 'Month': f"{best_m:02d}", 'Satellite': _modis_label(platform), 'LC_Source': lc_label,
            'Image_Date': f"{year}-{best_m:02d}-15 (night)",
            'Urban': round(urban_t, 2), 'Vegetative': round(rural_t, 2), 'UHI': round(urban_t - rural_t, 2),
            'Mean_LST': round(aoi_mean_t, 2),
            'Urban_Percent': up, 'Vegetative_Percent': vp, 'Other_Percent': op, 'LC_Year': lc_year}

# ---------- trend columns the app adds to its CSV (app lines 1705, 2222-2231) ----------
def sens_slope_and_p(x_vals, y_vals):
    mask = ~np.isnan(y_vals); x = x_vals[mask]; y = y_vals[mask]
    if y.size < 2: return None, None
    if _HAS_MK:
        res = mk.original_test(y); return float(res.slope), float(res.p)
    slope, _, _, _ = stats.theilslopes(y, x); _, p = stats.kendalltau(x, y)
    return float(slope), float(p)

def add_app_trend_cols(df, tag):
    ma = df['UHI'].astype(float).rolling(window=3, center=True, min_periods=1).mean()
    s, p = sens_slope_and_p(df['Year'].astype(int).to_numpy(), ma.to_numpy())
    df[f'UHI_MA_SenSlope_{tag}'] = s; df[f'UHI_MA_p_{tag}'] = p
    return df

## Run

In [ ]:
import time
MAX_TRIES = 4
def fname(city, run, tod): return os.path.join(OUT_DIR, f"{city}_{run}_{tod}.csv")
def n_years(run): return END_YEAR - RUNS[run]['start'] + 1
def is_done(city, run):
    return all(os.path.exists(fname(city, run, t)) and len(pd.read_csv(fname(city, run, t))) >= n_years(run) for t in ('DAY','NIGHT'))

def run_year(yr, aoi, cfg):
    for attempt in range(1, MAX_TRIES + 1):
        try:
            d, m = process_modis_day_and_best_month(yr, aoi, cfg['months'], cfg['mcd'], cfg['platform'], cfg['veg'])
            if d is None: return None, None
            return d, process_modis_night_for_month(yr, aoi, m, cfg['mcd'], cfg['platform'], cfg['veg'])
        except Exception as e:
            print(f"   {yr}: try {attempt} failed ({str(e)[:60]})"); time.sleep(30 * attempt)
    raise RuntimeError(f"{yr} failed")

aoi_cache = {}
for city, state in CITIES:
    if city not in aoi_cache:
        aoi_cache[city] = build_city_aoi(city, STATE_FIPS[state], "Recommended")
    aoi, urb_km2, buf_km = aoi_cache[city]
    for run, cfg in RUNS.items():
        if is_done(city, run): continue
        print(f"{city} | {run} | buffer {buf_km:.0f} km")
        day, night = [], []
        try:
            for yr in range(cfg['start'], END_YEAR + 1):
                d, n = run_year(yr, aoi, cfg)
                if d: day.append(d)
                if n: night.append(n)
        except Exception as e:
            print("   NOT SAVED:", e); continue
        if len(day) < n_years(run) or len(night) < n_years(run):
            print(f"   NOT SAVED: {len(day)} day / {len(night)} night years"); continue
        add_app_trend_cols(pd.DataFrame(day), 'Day').to_csv(fname(city, run, 'DAY'), index=False)
        add_app_trend_cols(pd.DataFrame(night), 'Night').to_csv(fname(city, run, 'NIGHT'), index=False)


## Summary vs default (comments 1, 3, 4) and trend-test comparison (comment 2)

In [ ]:
def series(city, run, tod):
    d = pd.read_csv(fname(city, run, tod)); return d.set_index('Year')['UHI'].astype(float)

def trend(y):
    yrs = np.asarray(y.index, float); v = y.values
    o = mk.original_test(v); h = mk.hamed_rao_modification_test(v)
    sm = pd.Series(v).rolling(3, center=True, min_periods=1).mean().values
    return dict(slope=stats.theilslopes(v, yrs)[0]*10, p_mk=o.p, p_hr=h.p, p_smoothed=mk.original_test(sm).p)

rows, trows = [], []
for city, _ in CITIES:
    for tod in ('DAY', 'NIGHT'):
        if not all(is_done(city, r) for r in RUNS): continue
        S = {r: series(city, r, tod) for r in RUNS}
        S['S1_jja_mean'] = (S['F6_june'] + S['F7_july'] + S['F8_aug']) / 3
        base = S['B0_default']; tb = trend(base)
        trows.append(dict(City=city, ToD=tod, **{k: round(v, 3) for k, v in tb.items()}))
        for r, s in S.items():
            if r == 'B0_default': continue
            yrs = base.index.intersection(s.index); b, x = base[yrs], s[yrs]
            t = trend(s)
            rows.append(dict(Test=r, City=city, ToD=tod, n_years=len(yrs),
                             mean_diff=(x - b).mean(), mean_abs_diff=(x - b).abs().mean(),
                             r_pearson=np.corrcoef(b, x)[0, 1],
                             trend_diff=t['slope'] - trend(b)['slope'],
                             same_sign_trend=np.sign(t['slope']) == np.sign(trend(b)['slope'])))
city_tab = pd.DataFrame(rows).round(3)
city_tab.to_csv(os.path.join(SUM_DIR, f"option_sensitivity_by_city_set{SET}.csv"), index=False)
test_tab = city_tab.groupby(['Test', 'ToD'])[['mean_diff', 'mean_abs_diff', 'r_pearson', 'trend_diff']].mean().round(2)
test_tab.to_csv(os.path.join(SUM_DIR, f"option_sensitivity_summary_set{SET}.csv"))
trend_tab = pd.DataFrame(trows); trend_tab.to_csv(os.path.join(SUM_DIR, f"trend_test_comparison_set{SET}.csv"), index=False)
display(test_tab); display(trend_tab)


In [ ]:
# Both sets side by side (run after Set A and Set B summaries exist)
import glob
tabs = []
for st in ("A", "B"):
    p = os.path.join(SUM_DIR, f"option_sensitivity_summary_set{st}.csv")
    if os.path.exists(p): tabs.append(pd.read_csv(p).assign(Set=st))
if len(tabs) == 2:
    both = pd.concat(tabs).pivot_table(index=['Test', 'ToD'], columns='Set',
                                       values=['mean_diff', 'mean_abs_diff', 'r_pearson', 'trend_diff']).round(2)
    both.to_csv(os.path.join(SUM_DIR, "option_sensitivity_setA_vs_setB.csv")); display(both)


Units: SUHI and differences in °C; slopes in °C per decade.